# Keep Working — Pomodoro Template

A Jupyter notebook implementing a Pomodoro-style productivity tool called "Keep Working" with timer logic, persistence, notifications, visualizations, VS Code integration examples, and unit tests.

Created: 2026-06-01


In [12]:
# Section 1 — Setup & Imports

# Standard, data, UI and OS/notification imports
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import sys
import os
import time
import asyncio
import json
import sqlite3
import datetime
from dataclasses import dataclass, asdict
from pathlib import Path
import logging
logging.basicConfig(level=logging.INFO)

# Data & plotting

# UI
try:
    import ipywidgets as widgets
except Exception:
    widgets = None

# Notification helper fallback
try:
    from plyer import notification
except Exception:
    notification = None

print('imports ready')

imports ready


In [13]:
# Section 2 — Configuration & Constants

# Defaults (seconds)
WORK_DURATION = 25 * 60  # 25 minutes
SHORT_BREAK = 5 * 60     # 5 minutes
LONG_BREAK = 15 * 60     # 15 minutes
CYCLES_PER_LONG_BREAK = 4

CONFIG_PATH = Path('.').resolve() / 'notebooks' / 'keep_working_config.json'


def load_config(path=CONFIG_PATH):
    if path.exists():
        return json.loads(path.read_text())
    return {}


def save_config(cfg, path=CONFIG_PATH):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(cfg, indent=2))


print('defaults set')

defaults set


In [14]:
# Section 3 — Task Data Model & Persistence

@dataclass
class SessionRecord:
    task_name: str
    start_ts: str
    end_ts: str
    duration_s: int
    kind: str  # 'work'|'short_break'|'long_break'


# JSON persistence
HISTORY_JSON = Path('notebooks') / 'keep_working_history.json'
HISTORY_DB = Path('notebooks') / 'keep_working_history.db'


def append_json_record(rec: SessionRecord, path=HISTORY_JSON):
    path.parent.mkdir(parents=True, exist_ok=True)
    arr = []
    if path.exists():
        arr = json.loads(path.read_text())
    arr.append(asdict(rec))
    path.write_text(json.dumps(arr, indent=2))

# Tiny SQLite persistence


def init_db(path=HISTORY_DB):
    conn = sqlite3.connect(str(path))
    cur = conn.cursor()
    cur.execute('''CREATE TABLE IF NOT EXISTS sessions(id INTEGER PRIMARY KEY, task_name TEXT, start_ts TEXT, end_ts TEXT, duration_s INTEGER, kind TEXT)''')
    conn.commit()
    conn.close()


def append_db_record(rec: SessionRecord, path=HISTORY_DB):
    init_db(path)
    conn = sqlite3.connect(str(path))
    cur = conn.cursor()
    cur.execute('INSERT INTO sessions(task_name, start_ts, end_ts, duration_s, kind) VALUES (?,?,?,?,?)',
                (rec.task_name, rec.start_ts, rec.end_ts, rec.duration_s, rec.kind))
    conn.commit()
    conn.close()


print('persistence helpers ready')

persistence helpers ready


In [15]:
# Section 4 — Pomodoro Timer Core (synchronous)

class PomodoroTimer:
    def __init__(self, work=WORK_DURATION, short=SHORT_BREAK, long=LONG_BREAK, cycles_per_long=CYCLES_PER_LONG_BREAK):
        self.work = work
        self.short = short
        self.long = long
        self.cycles_per_long = cycles_per_long
        self._running = False
        self._paused = False
        self._state = 'idle'  # idle, work, short_break, long_break
        self._cycle_count = 0
        self._start_ts = None

    def start_work(self):
        self._state = 'work'
        self._running = True
        self._paused = False
        self._start_ts = time.time()
        print('Work started')

    def pause(self):
        if self._running and not self._paused:
            self._paused = True
            print('paused')

    def resume(self):
        if self._running and self._paused:
            self._paused = False
            print('resumed')

    def reset(self):
        self._running = False
        self._paused = False
        self._state = 'idle'
        self._start_ts = None
        print('reset')


# Blocking demo (short durations for demo)
if False:
    t = PomodoroTimer(work=5, short=3, long=7, cycles_per_long=2)
    t.start_work()
    time.sleep(1)
    t.pause()
    time.sleep(0.5)
    t.resume()
    t.reset()

In [16]:
# Section 5 — Async Timer & ipywidgets Controls

# Async non-blocking timer using asyncio with notebook UI callbacks
class AsyncPomodoro:
    def __init__(self, work=WORK_DURATION, short=SHORT_BREAK, long=LONG_BREAK, cycles_per_long=CYCLES_PER_LONG_BREAK):
        self.work = work
        self.short = short
        self.long = long
        self.cycles_per_long = cycles_per_long
        self._task = None
        self._cancel = False
        self.state = 'idle'
        self.remaining = 0
        self.current_kind = None
        self._on_finish = None  # optional callback
        self.notify = False  # enable desktop notifications
        self.sound = False   # enable sound alerts

    async def _countdown(self, seconds, kind='work'):
        self.current_kind = kind
        self.remaining = seconds
        self.state = kind
        start = time.time()
        try:
            while self.remaining > 0 and not self._cancel:
                await asyncio.sleep(1)
                self.remaining = max(0, seconds - int(time.time() - start))
            if not self._cancel:
                # finished normally; trigger on_finish then optional notifications/sound
                if self._on_finish:
                    try:
                        self._on_finish(kind)
                    except Exception as e:
                        print('on_finish callback error', e)
                if self.notify:
                    try:
                        send_notification('Keep Working', f'{kind} finished')
                    except Exception:
                        pass
                if self.sound:
                    try:
                        play_sound()
                    except Exception:
                        pass
        finally:
            self._cancel = False

    def start(self, loop, seconds=None, kind='work', on_finish=None):
        if seconds is None:
            seconds = self.work if kind == 'work' else self.short
        self._cancel = False
        self._on_finish = on_finish
        # create task on provided loop
        self._task = loop.create_task(self._countdown(seconds, kind=kind))

    def stop(self):
        # request cancellation
        self._cancel = True
        if self._task:
            try:
                self._task.cancel()
            except Exception:
                pass

    def is_running(self):
        return self._task is not None and not self._task.done()


# Notebook UI wiring (creates buttons and a live label)
async_pom = AsyncPomodoro()
loop = None
try:
    # Jupyter running loop
    loop = asyncio.get_running_loop()
except RuntimeError:
    loop = asyncio.new_event_loop()
    asyncio.set_event_loop(loop)

# updater coroutine to refresh the label every second


async def _updater(label_widget):
    while True:
        try:
            if async_pom.current_kind:
                mins, secs = divmod(async_pom.remaining, 60)
                label_widget.value = f'{async_pom.current_kind}: {mins:02d}:{secs:02d}'
            else:
                label_widget.value = 'Idle'
            await asyncio.sleep(1)
        except asyncio.CancelledError:
            break


def _on_finish(kind):
    # log finished session and append to history
    end = datetime.datetime.now(datetime.timezone.utc)
    start = end - \
        datetime.timedelta(
            seconds=(async_pom.work if kind == 'work' else async_pom.short))
    rec = SessionRecord(task_name='Unnamed', start_ts=start.isoformat(), end_ts=end.isoformat(
    ), duration_s=(async_pom.work if kind == 'work' else async_pom.short), kind=kind)
    append_json_record(rec)
    append_db_record(rec)
    print('logged', kind)


if widgets:
    start_btn = widgets.Button(description='Start Work')
    pause_btn = widgets.Button(description='Stop')
    reset_btn = widgets.Button(description='Reset')
    skip_btn = widgets.Button(description='Skip')
    label = widgets.Label('Idle')
    box = widgets.HBox([start_btn, pause_btn, reset_btn, skip_btn, label])
    display(box)

    # Start button callback
    def on_start(b):
        if not async_pom.is_running():
            async_pom.start(loop, kind='work', on_finish=_on_finish)
            # ensure updater is running
            try:
                # schedule updater if not already
                if not hasattr(on_start, '_updater_task') or on_start._updater_task.done():
                    on_start._updater_task = loop.create_task(_updater(label))
            except Exception:
                pass

    # Stop button callback (cancels current session)
    def on_stop(b):
        async_pom.stop()
        label.value = 'Stopped'

    # Reset callback clears state
    def on_reset(b):
        async_pom.stop()
        async_pom.current_kind = None
        async_pom.remaining = 0
        label.value = 'Idle'

    # Skip moves to short break
    def on_skip(b):
        async_pom.stop()
        async_pom.start(loop, seconds=async_pom.short,
                        kind='short_break', on_finish=_on_finish)

    start_btn.on_click(on_start)
    pause_btn.on_click(on_stop)
    reset_btn.on_click(on_reset)
    skip_btn.on_click(on_skip)

print('async timer UI wired')

# In non-widget environments expose async_pom and loop for manual use
__async_pom__ = async_pom
__async_loop__ = loop

async timer UI wired


In [17]:
# Section 6 — Notifications & Sound Alerts

def send_notification(title, message):
    if notification:
        try:
            notification.notify(title=title, message=message)
        except Exception:
            print('notification fallback:', title, message)
    else:
        print('notification:', title, message)

# Sound alert fallback: try to use simple beep


def play_sound():
    try:
        # cross-platform simple beep
        print('\a')
    except Exception:
        pass


print('notification helpers ready')

notification helpers ready


In [18]:
# Section 7 — Logging Sessions & Saving History

# Example: log a completed work session
_now = datetime.datetime.now(datetime.timezone.utc)
rec = SessionRecord(task_name='Example Task', start_ts=_now.isoformat(), end_ts=(
    _now+datetime.timedelta(minutes=25)).isoformat(), duration_s=25*60, kind='work')
append_json_record(rec)
append_db_record(rec)
print('sample record appended')

sample record appended


In [19]:
# Section 8 — Visualizing Productivity

# Load JSON history into pandas and plot daily counts
if HISTORY_JSON.exists():
    hist = pd.read_json(HISTORY_JSON)
else:
    # fallback read from sqlite
    conn = sqlite3.connect(str(HISTORY_DB))
    hist = pd.read_sql_query('SELECT * FROM sessions', conn)
    conn.close()

if not hist.empty:
    hist['start_dt'] = pd.to_datetime(hist['start_ts'])
    hist['date'] = hist['start_dt'].dt.date
    daily = hist.groupby('date').size().reset_index(name='sessions')
    plt.figure(figsize=(8, 3))
    sns.barplot(data=daily, x='date', y='sessions')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()
else:
    print('no history to plot')

ValueError: unconverted data remains when parsing with format "%Y-%m-%dT%H:%M:%S.%f": "+00:00". You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [ ]:
# Section 9 — Exporting / Importing CSV & JSON

def export_history_csv(path=Path('artifacts')/'keep_working_history.csv'):
    path.parent.mkdir(parents=True, exist_ok=True)
    if HISTORY_JSON.exists():
        df = pd.read_json(HISTORY_JSON)
        df.to_csv(path, index=False)
        print('exported', path)
    else:
        print('no JSON history to export')

# Import with simple validation


def import_history_json(path):
    data = json.loads(Path(path).read_text())
    # basic schema validation
    for r in data:
        if not all(k in r for k in ('task_name', 'start_ts', 'end_ts', 'duration_s', 'kind')):
            raise ValueError('invalid record')
    # append safely
    existing = []
    if HISTORY_JSON.exists():
        existing = json.loads(HISTORY_JSON.read_text())
    merged = existing + data
    HISTORY_JSON.write_text(json.dumps(merged, indent=2))
    print('imported', len(data), 'records')

In [ ]:
# Section 10 — Automated Break Scheduling & Rules Engine

# Simple rules engine: increase break if > N consecutive sessions
RULES = {'fatigue_threshold': 3, 'extra_break_secs': 60}


def adjusted_break(consecutive_sessions, base_short=SHORT_BREAK):
    if consecutive_sessions >= RULES['fatigue_threshold']:
        return base_short + RULES['extra_break_secs']
    return base_short


print('rules engine ready')

rules engine ready


In [ ]:
# Section 11 — Integrating with VS Code (examples)

# Example: write a log file that could be read by a VS Code task or extension
VS_LOG = Path('.vscode') / 'keep_working.log'
VS_LOG.parent.mkdir(parents=True, exist_ok=True)
VS_LOG.write_text('Keep Working log initialized\n')
print('wrote', VS_LOG)

# Notebook UI: expose cycle settings and run/stop controls
if widgets:
    task_input = widgets.Text(value='Unnamed', description='Task:')
    work_input = widgets.IntText(value=25*60, description='Work(s):')
    short_input = widgets.IntText(value=5*60, description='Short(s):')
    long_input = widgets.IntText(value=15*60, description='Long(s):')
    cycles_input = widgets.IntText(value=4, description='Cycles/Long:')
    notify_chk = widgets.Checkbox(value=False, description='Notify')
    sound_chk = widgets.Checkbox(value=False, description='Sound')
    repeat_chk = widgets.Checkbox(value=False, description='Repeat')
    status_input = widgets.Text(value=str(
        Path('notebooks')/'keep_working_status.json'), description='StatusFile:')
    run_btn = widgets.ToggleButton(value=False, description='Run')
    stop_btn = widgets.Button(description='Stop')
    export_btn = widgets.Button(description='Export systemd')
    hbox = widgets.VBox([task_input, widgets.HBox([work_input, short_input, long_input, cycles_input]), widgets.HBox(
        [notify_chk, sound_chk, repeat_chk]), status_input, widgets.HBox([run_btn, stop_btn, export_btn])])
    display(hbox)

    # Runner that implements the same loop as CLI but non-blocking via asyncio
    _runner_task = None

    async def _runner():
        async_pom.notify = notify_chk.value
        async_pom.sound = sound_chk.value
        cycle = 0
        loop = __async_loop__
        while run_btn.value:
            cycle += 1
            async_pom.start(loop, seconds=work_input.value,
                            kind='work', on_finish=_on_finish)
            # wait until done or cancelled
            while async_pom.is_running():
                await asyncio.sleep(0.5)
            if not run_btn.value:
                break
            # decide break
            if cycles_input.value > 0 and (cycle % cycles_input.value) == 0:
                bsecs = long_input.value
                bkind = 'long_break'
            else:
                bsecs = short_input.value
                bkind = 'short_break'
            async_pom.start(loop, seconds=bsecs, kind=bkind,
                            on_finish=_on_finish)
            while async_pom.is_running():
                await asyncio.sleep(0.5)
        print('runner exiting')

    def _on_run_toggle(change):
        global _runner_task
        if change['new']:
            _runner_task = __async_loop__.create_task(_runner())
        else:
            async_pom.stop()

    def _on_stop_click(b):
        run_btn.value = False
        async_pom.stop()

    def _export_systemd_clicked(b):
        # write a systemd service file using current widget values
        svc_path = Path(status_input.value).parent / \
            'keep_working_from_notebook.service'
        try:
            python_exec = sys.executable
            workspace_dir = str(Path.cwd())
            launcher = str(Path(workspace_dir) / 'notebooks' /
                           'keep_working_launcher.py')
            args = [python_exec, launcher, '--task', task_input.value, '--work', str(work_input.value), '--short', str(
                short_input.value), '--long', str(long_input.value), '--cycles-per-long', str(cycles_input.value)]
            if notify_chk.value:
                args.append('--notify')
            if sound_chk.value:
                args.append('--sound')
            if repeat_chk.value:
                args.append('--repeat')
            import shlex
            import getpass
            exec_cmd = ' '.join(shlex.quote(a) for a in args)
            content = (
                "[Unit]\n"
                "Description=Keep Working (notebook export)\n"
                "\n"
                "[Service]\n"
                "Type=simple\n"
                f"WorkingDirectory={workspace_dir}\n"
                f"ExecStart={exec_cmd}\n"
                "Restart=on-failure\n"
                f"User={getpass.getuser()}\n"
                "\n"
                "[Install]\n"
                "WantedBy=default.target\n"
            )
            svc_path.write_text(content)
            print('Wrote', svc_path)
        except Exception as e:
            print('Failed to export systemd:', e)

    run_btn.observe(_on_run_toggle, names='value')
    stop_btn.on_click(_on_stop_click)
    export_btn.on_click(_export_systemd_clicked)

NameError: name 'Path' is not defined

In [ ]:
# Section 12 — Unit Tests for Timer Logic and Persistence

# Minimal pytest-style tests (uses the launcher module which is importable)
# Write to repo root tests/ so pytest discovers them from the project root
TESTS_DIR = Path(__file__).resolve().parent.parent / 'tests' / \
    'keep_working' if '__file__' in dir() else Path.cwd().parent / \
    'tests' / 'keep_working'
TESTS_DIR.mkdir(parents=True, exist_ok=True)

test_code = '''"""Tests for Keep Working Pomodoro timer and persistence."""
import json
import time
import sys
from pathlib import Path

# Ensure the launcher is importable
sys.path.insert(0, str(Path(__file__).resolve().parent.parent.parent))
from notebooks.keep_working_launcher import SessionRecord, append_json_record


def test_session_record_fields():
    rec = SessionRecord(task_name='t', start_ts='a', end_ts='b', duration_s=10, kind='work')
    assert rec.task_name == 't'
    assert rec.kind == 'work'
    assert rec.duration_s == 10


def test_persistence_write(tmp_path):
    p = tmp_path / 'hist.json'
    rec = SessionRecord(task_name='t', start_ts='a', end_ts='b', duration_s=10, kind='work')
    append_json_record(rec, path=p)
    assert p.exists()
    data = json.loads(p.read_text())
    assert len(data) == 1
    assert data[0]['task_name'] == 't'


def test_persistence_append(tmp_path):
    p = tmp_path / 'hist.json'
    rec1 = SessionRecord(task_name='t1', start_ts='a', end_ts='b', duration_s=10, kind='work')
    rec2 = SessionRecord(task_name='t2', start_ts='c', end_ts='d', duration_s=20, kind='short_break')
    append_json_record(rec1, path=p)
    append_json_record(rec2, path=p)
    data = json.loads(p.read_text())
    assert len(data) == 2
    assert data[1]['task_name'] == 't2'
'''
(TESTS_DIR / 'test_keep_working.py').write_text(test_code)
print('wrote tests to', TESTS_DIR)

# Example command to run tests (copy-paste):
print('\nRun: .venv/bin/python -m pytest tests/keep_working -q')

wrote tests to /workspaces/Aria/tests/keep_working

Run: .venv/bin/python -m pytest tests/keep_working -q
